In [5]:
import numpy as np
from rdkit import Chem
from pathlib import Path
import pandas as pd

In [2]:
atomic_features: list[str] = [
    "atomic_fukui_minus",
    "atomic_fukui_plus",
    "atomic_dipole_norm",
    "atomic_quadrupole_principal_invariant_2",
    "atomic_quadrupole_principal_invariant_3",
    "atomic_polarizability_mean",
    "atomic_polarizability_anisotropy",
    "atomic_sasa",
    "partial_charge",
]

In [3]:
bond_features: list[str] = [
    "bond_length",
]

In [7]:
from scipy.constants import (
    Avogadro,  # 1/mol
    Boltzmann,  # in J/K
)

def boltzmann_weights(G: np.ndarray, T: float = 300.0) -> np.ndarray:
    k_B: float = Boltzmann * Avogadro * 0.000239005736
    delta_G = G - G.min()
    factors = np.exp(-delta_G / (k_B * T))
    return factors / factors.sum()

In [8]:
from ml_enhance import QuantumFPFileLoader

loader = QuantumFPFileLoader(Path("../data/QuantumFP/QFP_output"))
filelist = loader.list_output_files()

In [9]:
from collections.abc import Generator

def stream_conformer_df(
    file: Path,
    loader: QuantumFPFileLoader,
) -> Generator[pd.DataFrame, None, None]:
    for df in loader.stream_conformer_dataframe(file):
        df["gibbs_free_energy_300K"] = df["gibbs_free_energy"].map(lambda x: x[1][1])
        yield df

In [ ]:
def extract_bond_features(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: implement bond feature extraction
    selected_df = df[["original_smiles", "gibbs_free_energy_300K", *bond_features]]
    object_cols = selected_df.select_dtypes(include="object").columns.to_list()
    exploded_df = selected_df.explode(object_cols)

    G = selected_df["gibbs_free_energy_300K"].unique()
    weights = boltzmann_weights(G)

    arr = np.array(exploded_df[object_cols].values.tolist())  # (n_conformers * n_atoms, n_features, 2)
    atom_map_idx = arr[:, 0, 0].astype(int)
    values = arr[:, :, 1].astype(float)

    return exploded_df

In [23]:
l = []
# for file in filelist[:2]:
for df in stream_conformer_df(filelist[1002], loader):
    tdf = extract_bond_features(df)
    # l.append(tdf)

# combo_df = pd.concat(l, ignore_index=True)

In [24]:
tdf #[["atom", "partial_charge"]]

,original_smiles,gibbs_free_energy_300K,bond_length
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,11.109862,"[1, 2, 2.6515934831733237]"
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,11.109862,"[2, 3, 2.884522293692259]"
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,11.109862,"[3, 4, 3.442128284432925]"
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,11.109862,"[1, 5, 1.821851208812429]"
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,11.109862,"[2, 6, 2.0809694268189594]"
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,11.109862,"[2, 7, 2.0644596324771336]"
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,11.109862,"[3, 8, 2.063831886752403]"
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,11.109862,"[3, 9, 2.0589483487637694]"
0,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,11.109862,"[4, 10, 2.51800947907787]"
1,[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H...,10.696520,"[1, 2, 2.6450685965743155]"


In [ ]:
tdf.loc[0, "original_smiles"]

'[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H:6])[H:7])[H:5]'

In [ ]:
mol = Chem.MolFromSmiles('[O:1]([C:2]([C:3]([S:4][H:10])([H:8])[H:9])([H:6])[H:7])[H:5]', sanitize=False)

In [ ]:
for atom in mol.GetAtoms():
    idx = atom.GetIdx()
    print(idx, atom.GetAtomMapNum(), atom.GetSymbol())


0 1 O
1 2 C
2 3 C
3 4 S
4 10 H
5 8 H
6 9 H
7 6 H
8 7 H
9 5 H


In [ ]:
mol.GetNumBonds()

9

In [ ]:
for bond in mol.GetBonds():
    print(bond.GetIdx(), bond.GetBeginAtom().GetAtomMapNum(), bond.GetEndAtom().GetAtomMapNum())

0 1 2
1 2 3
2 3 4
3 4 10
4 3 8
5 3 9
6 2 6
7 2 7
8 1 5
